# Train leaf instance-segmentation candidates

Notebook này chứa training loop thật cho các YOLO segmentation candidate. DVC thực thi notebook bằng Papermill; notebook chỉ ghi checkpoint và metrics theo contract mà `pipeline.py select` sử dụng.

In [ ]:
params_path = "params.yaml"


In [ ]:
from pathlib import Path
from typing import Any
import json
import os
import shutil
import tempfile

project_root = Path.cwd().resolve()
if not (project_root / "pipeline.py").exists():
    project_root = project_root.parent
if not (project_root / "pipeline.py").exists():
    raise RuntimeError("Run this notebook from the CoffeeLeaf-AI repository")
os.chdir(project_root)

import torch
import yaml
from ultralytics import YOLO

from pipeline import (
    ROOT,
    load_config,
    project_path,
    reset_dir,
    seed_everything,
    write_csv,
    write_json,
)

print(f"Repository: {ROOT}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


In [ ]:
def nested_attr(obj: Any, path: str, default: float = 0.0) -> float:
    current = obj
    for name in path.split("."):
        if current is None:
            return default
        current = getattr(current, name, None)
    try:
        return float(current)
    except (TypeError, ValueError):
        return default


def runtime_yolo_yaml(segment_root: Path) -> Path:
    payload = {
        "path": str(segment_root.resolve()),
        "train": "images/train",
        "val": "images/val",
        "test": "images/test",
        "names": {0: "leaf"},
    }
    with tempfile.NamedTemporaryFile(
        "w", suffix=".yaml", encoding="utf-8", delete=False
    ) as handle:
        yaml.safe_dump(payload, handle, sort_keys=False)
        return Path(handle.name)


def resolve_device(value: Any) -> Any:
    return None if str(value).lower() == "auto" else value


In [ ]:
config = load_config(params_path)
seed = int(config["seed"])
seed_everything(seed)
settings = config["segmentation"]
worker_count = 0 if os.name == "nt" else int(settings["workers"])
segment_root = project_path(config["data"]["processed_dir"]) / "segmentation"
required_splits = [segment_root / "images" / split for split in ("train", "val", "test")]
missing = [str(path) for path in required_splits if not path.exists()]
if missing:
    raise RuntimeError("Prepared segmentation data is missing. Run `dvc repro prepare`. Missing: " + ", ".join(missing))

print(json.dumps(settings, indent=2))
print(f"Ultralytics workers: {worker_count}")


In [ ]:
output_dir = reset_dir(ROOT / "models" / "segmenters")
data_yaml = runtime_yolo_yaml(segment_root)
metrics_by_candidate: dict[str, dict[str, Any]] = {}
rows: list[dict[str, Any]] = []

try:
    for candidate, pretrained_weights in settings["candidates"].items():
        print(f"\n=== Training segmenter: {candidate} ===")
        source = str(pretrained_weights)
        if not bool(settings.get("pretrained", True)) and source.endswith(".pt"):
            source = source[:-3] + ".yaml"
        model = YOLO(source)
        with tempfile.TemporaryDirectory(prefix=f"coffee-{candidate}-") as run_dir:
            train_args: dict[str, Any] = {
                "data": str(data_yaml),
                "epochs": int(settings["epochs"]),
                "imgsz": int(settings["image_size"]),
                "batch": int(settings["batch_size"]),
                "patience": int(settings["patience"]),
                "workers": worker_count,
                "seed": seed,
                "deterministic": bool(settings["deterministic"]),
                "project": run_dir,
                "name": candidate,
                "exist_ok": True,
                "verbose": True,
            }
            device = resolve_device(settings.get("device", "auto"))
            if device is not None:
                train_args["device"] = device
            model.train(**train_args)
            best_path = Path(model.trainer.best)
            if not best_path.exists():
                raise RuntimeError(f"Ultralytics did not create best weights for {candidate}")
            target = output_dir / f"{candidate}.pt"
            shutil.copy2(best_path, target)

        best_model = YOLO(str(target))
        validation = best_model.val(
            data=str(data_yaml), split="val", imgsz=int(settings["image_size"]),
            batch=int(settings["batch_size"]), workers=worker_count, verbose=False,
        )
        test_result = best_model.val(
            data=str(data_yaml), split="test", imgsz=int(settings["image_size"]),
            batch=int(settings["batch_size"]), workers=worker_count, verbose=False,
        )
        speed = getattr(validation, "speed", {}) or {}
        values = {
            "map50_95": nested_attr(validation, "seg.map"),
            "map50": nested_attr(validation, "seg.map50"),
            "precision": nested_attr(validation, "seg.mp"),
            "recall": nested_attr(validation, "seg.mr"),
            "test_map50_95": nested_attr(test_result, "seg.map"),
            "test_map50": nested_attr(test_result, "seg.map50"),
            "test_precision": nested_attr(test_result, "seg.mp"),
            "test_recall": nested_attr(test_result, "seg.mr"),
            "latency_ms": float(speed.get("inference", 0.0)),
            "size_mb": target.stat().st_size / (1024 * 1024),
            "image_size": int(settings["image_size"]),
        }
        metrics_by_candidate[candidate] = values
        rows.append({
            "candidate": candidate,
            **{key: values[key] for key in ("map50_95", "recall", "latency_ms", "size_mb")},
        })
        del model, best_model, validation, test_result
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
finally:
    data_yaml.unlink(missing_ok=True)

write_json(ROOT / "metrics" / "segmenters.json", {"candidates": metrics_by_candidate})
write_csv(ROOT / "metrics" / "segmenters.csv", rows)


In [ ]:
result = json.loads((ROOT / "metrics" / "segmenters.json").read_text(encoding="utf-8"))
result
